# 03 — Customer Segmentation (K-Means RFM)

**Source:** `outputs/rfm_with_labels.parquet` (2,835 customers, 23.4% churn)  
**Output:** `outputs/rfm_with_labels.parquet` (overwritten + `segment_label`), `outputs/segment_profiles.csv`

| Section | What |
|---|---|
| 1 — Preprocessing | log1p transform on frequency & monetary; StandardScaler |
| 2 — Optimal K | Elbow + silhouette k=2..8, dual-axis chart |
| 3 — Fit K-Means | k=4, composite-rank segment naming |
| 4 — Visualize | PCA scatter, count bar, churn-rate-by-segment bar |
| 5 — Save | Parquet + segment_profiles.csv |

**Segment palette:** Champions (green) → Loyal (blue) → At-Risk (orange) → Lost (red)

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [2]:
INPUT_PATH   = Path("../outputs/rfm_with_labels.parquet")
OUTPUT_PATH  = Path("../outputs/rfm_with_labels.parquet")
PROFILE_PATH = Path("../outputs/segment_profiles.csv")

assert INPUT_PATH.exists(), f"Run notebook 02 first — {INPUT_PATH} not found"

df = pd.read_parquet(INPUT_PATH)
print(f"Loaded: {df.shape[0]:,} customers × {df.shape[1]} features")
print(f"Churn rate: {df['churned'].mean()*100:.1f}%")
df.head()

Loaded: 2,835 customers × 10 features
Churn rate: 23.4%


,customer_id,recency,frequency,monetary,aov,purchase_velocity,days_as_customer,unique_products,unique_countries,churned
0,12346,156,11,372.86,33.896364,0.055838,196,26,1,0
1,12349,34,3,2671.14,890.380000,0.016393,182,90,1,0
2,12352,2,2,343.80,171.900000,0.111111,17,18,1,0
3,12356,7,3,3560.30,1186.766667,0.066667,44,68,1,0
4,12357,15,2,12079.99,6039.995000,2.000000,0,165,1,0


---
## Section 1 — Preprocessing for Clustering

Features used: **recency, frequency, monetary** — pure RFM, no leakage from extended features.

Transforms applied before scaling:
- `frequency` → `log1p(frequency)`: heavy right skew (max=398, median=3)
- `monetary`  → `log1p(monetary)`: extreme right skew (max=£580k, median=£867)
- `recency` is left untransformed — already roughly continuous and measured in days

`StandardScaler` is fit on all 2,835 customers (segmentation uses the full population).

In [3]:
X = df[["recency", "frequency", "monetary"]].copy()
X["frequency"] = np.log1p(X["frequency"])
X["monetary"]  = np.log1p(X["monetary"])

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix : {X_scaled.shape[0]:,} rows × {X_scaled.shape[1]} features")
print("\nPost-transform stats (before scaling):")
print(X.describe().round(3))

Feature matrix : 2,835 rows × 3 features

Post-transform stats (before scaling):
        recency  frequency  monetary
count  2835.000   2835.000  2835.000
mean     59.841      1.690     7.155
std      69.039      0.603     1.071
min       1.000      1.099     3.221
25%      12.000      1.099     6.430
50%      34.000      1.609     7.109
75%      78.000      1.946     7.765
max     365.000      5.288    12.680


---
## Section 2 — Find Optimal K

Run K-Means for k = 2 to 8. Track two metrics:
- **Inertia (elbow method):** sum of squared distances to cluster centres — look for the "elbow" where adding more clusters gives diminishing returns
- **Silhouette score:** measures how well each point fits its cluster vs. neighbours (higher = better, range −1..1)

In [4]:
K_RANGE     = range(2, 9)
inertias    = []
silhouettes = []

for k in K_RANGE:
    km  = KMeans(n_clusters=k, n_init=10, random_state=42)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, lbl, random_state=42))

ks = list(K_RANGE)
print(f"{'k':>3}  {'Inertia':>12}  {'Silhouette':>12}")
for k, ine, sil in zip(ks, inertias, silhouettes):
    print(f"{k:>3}  {ine:>12.1f}  {sil:>12.4f}")

  k       Inertia    Silhouette
  2        5031.5        0.3781
  3        3293.4        0.4098
  4        2512.9        0.3633
  5        2154.6        0.3158
  6        1889.1        0.3144
  7        1709.1        0.2899
  8        1546.8        0.2938


In [5]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ks, y=inertias,
    name="Inertia (elbow)",
    mode="lines+markers",
    marker=dict(symbol="circle", size=9),
    line=dict(color="#2196F3", width=2),
    yaxis="y1",
))
fig.add_trace(go.Scatter(
    x=ks, y=silhouettes,
    name="Silhouette score",
    mode="lines+markers",
    marker=dict(symbol="diamond", size=9),
    line=dict(color="#FF9800", width=2),
    yaxis="y2",
))

fig.update_layout(
    title="K-Means: Elbow (inertia) and Silhouette score vs k",
    xaxis=dict(title="Number of clusters (k)", tickmode="array", tickvals=ks),
    yaxis=dict(title=dict(text="Inertia", font=dict(color="#2196F3"))),
    yaxis2=dict(
        title=dict(text="Silhouette score", font=dict(color="#FF9800")),
        overlaying="y",
        side="right",
    ),
    legend=dict(x=0.35, y=1.12, orientation="h"),
)
fig.show()

In [6]:
best_sil_k = ks[int(np.argmax(silhouettes))]
print(f"Data-driven recommendation : k = {best_sil_k}  (silhouette = {max(silhouettes):.4f})")
print(f"Final model choice         : k = 4")
print()
print("  k=4 is standard for RFM segmentation and yields four business-legible")
print("  archetypes: Champions, Loyal, At-Risk, Lost.")

Data-driven recommendation : k = 3  (silhouette = 0.4098)
Final model choice         : k = 4

  k=4 is standard for RFM segmentation and yields four business-legible
  archetypes: Champions, Loyal, At-Risk, Lost.


---
## Section 3 — Fit Final K-Means (k = 4)

Segments are named by **composite rank**: average of recency rank (ascending, lower=better)
and monetary rank (descending, higher=better). The cluster with the best composite score
becomes *Champions*, then *Loyal*, *At-Risk*, *Lost*.

In [7]:
FINAL_K  = 4
km_final = KMeans(n_clusters=FINAL_K, n_init=20, random_state=42)
df["cluster"] = km_final.fit_predict(X_scaled)

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index().to_string())

Cluster sizes:
cluster
0    1129
1     408
2     289
3    1009


In [8]:
# Profiles on original (untransformed) RFM scale for interpretability
profiles = (
    df.groupby("cluster")[["recency", "frequency", "monetary"]]
    .mean()
    .round(1)
    .reset_index()
)

# Composite rank: lower recency = better, higher monetary = better
profiles["recency_rank"]   = profiles["recency"].rank()                    # 1 = lowest (best)
profiles["monetary_rank"]  = profiles["monetary"].rank(ascending=False)    # 1 = highest (best)
profiles["composite_rank"] = (profiles["recency_rank"] + profiles["monetary_rank"]) / 2
profiles = profiles.sort_values("composite_rank").reset_index(drop=True)

SEGMENT_NAMES = ["Champions", "Loyal", "At-Risk", "Lost"]
name_map = dict(zip(profiles["cluster"], SEGMENT_NAMES))
df["segment_label"] = df["cluster"].map(name_map)

print("Segment profile (original RFM scale):")
print(
    profiles[["cluster", "recency", "frequency", "monetary"]]
    .assign(segment=SEGMENT_NAMES)
    [["segment", "recency", "frequency", "monetary"]]
    .to_string(index=False)
)

Segment profile (original RFM scale):
  segment  recency  frequency  monetary
Champions     17.5       22.4   14609.9
    Loyal     33.7        6.3    2486.2
  At-Risk    202.5        2.8    1019.9
     Lost     42.5        2.7     739.9


In [9]:
seg_profile = (
    df.groupby("segment_label")
    .agg(
        n_customers    = ("customer_id", "count"),
        mean_recency   = ("recency",     "mean"),
        mean_frequency = ("frequency",   "mean"),
        mean_monetary  = ("monetary",    "mean"),
        churn_rate_pct = ("churned",     lambda x: x.mean() * 100),
    )
    .round(1)
    .loc[SEGMENT_NAMES]
    .reset_index()
)

print("Full segment profile table:")
print(seg_profile.to_string(index=False))

Full segment profile table:
segment_label  n_customers  mean_recency  mean_frequency  mean_monetary  churn_rate_pct
    Champions          289          17.5            22.4        14609.9             4.5
        Loyal         1009          33.7             6.3         2486.2            11.1
      At-Risk          408         202.5             2.8         1019.9            49.0
         Lost         1129          42.5             2.7          739.9            30.0


---
## Section 4 — Visualize

In [10]:
pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1":           coords[:, 0],
    "PC2":           coords[:, 1],
    "segment_label": df["segment_label"].values,
    "monetary":      df["monetary"].values,
    "customer_id":   df["customer_id"].values,
})
pca_df["size"] = np.log1p(pca_df["monetary"])   # log1p prevents outlier from dominating

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100

fig = px.scatter(
    pca_df, x="PC1", y="PC2",
    color="segment_label",
    size="size",
    size_max=7,
    hover_data={"customer_id": True, "monetary": ":.0f", "size": False},
    category_orders={"segment_label": SEGMENT_NAMES},
    color_discrete_sequence=["#4CAF50", "#2196F3", "#FF9800", "#F44336"],
    title=f"PCA 2D projection — RFM segments (k={FINAL_K}), marker size = log(monetary)",
    labels={"PC1": f"PC1 ({var1:.1f}% var)", "PC2": f"PC2 ({var2:.1f}% var)"},
    opacity=0.7,
)
fig.show()

In [11]:
fig = px.bar(
    seg_profile, x="segment_label", y="n_customers",
    color="segment_label",
    category_orders={"segment_label": SEGMENT_NAMES},
    color_discrete_sequence=["#4CAF50", "#2196F3", "#FF9800", "#F44336"],
    text="n_customers",
    title="Customer count per segment",
    labels={"segment_label": "Segment", "n_customers": "Customers"},
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

In [12]:
fig = px.bar(
    seg_profile, x="segment_label", y="churn_rate_pct",
    color="segment_label",
    category_orders={"segment_label": SEGMENT_NAMES},
    color_discrete_sequence=["#4CAF50", "#2196F3", "#FF9800", "#F44336"],
    text=seg_profile["churn_rate_pct"].map(lambda p: f"{p:.1f}%"),
    title="Churn rate % per segment — segmentation meets churn model",
    labels={"segment_label": "Segment", "churn_rate_pct": "Churn rate (%)"},
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis_title="Churn rate (%)")
fig.show()

---
## Section 5 — Save

- **`outputs/rfm_with_labels.parquet`** — overwritten with `segment_label` column added (11 features total)
- **`outputs/segment_profiles.csv`** — one row per segment with mean RFM stats and churn rate

In [13]:
df_out = df.drop(columns=["cluster"])   # cluster is an implementation detail; keep only segment_label

df_out.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)
seg_profile.to_csv(PROFILE_PATH, index=False)

# Smoke-check
check = pd.read_parquet(OUTPUT_PATH)
assert "segment_label" in check.columns,                       "segment_label column missing"
assert "cluster" not in check.columns,                         "cluster column should not be saved"
assert set(check["segment_label"].unique()) == set(SEGMENT_NAMES), "unexpected segment names"
assert len(check) == len(df),                                  "row count mismatch"
assert check.shape[1] == 11,                                   f"expected 11 cols, got {check.shape[1]}"

print(f"Saved parquet → {OUTPUT_PATH.resolve()}")
print(f"Saved CSV     → {PROFILE_PATH.resolve()}")
print(f"Shape         : {check.shape[0]:,} customers × {check.shape[1]} features")
print(f"Columns       : {list(check.columns)}")
print()
print("Segment counts (final):")
print(check["segment_label"].value_counts().loc[SEGMENT_NAMES].to_string())
print()
print("All assertions passed.")

Saved parquet → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\rfm_with_labels.parquet
Saved CSV     → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\segment_profiles.csv
Shape         : 2,835 customers × 11 features
Columns       : ['customer_id', 'recency', 'frequency', 'monetary', 'aov', 'purchase_velocity', 'days_as_customer', 'unique_products', 'unique_countries', 'churned', 'segment_label']

Segment counts (final):
segment_label
Champions     289
Loyal        1009
At-Risk       408
Lost         1129

All assertions passed.
